In [13]:
!pip install qdrant-client

In [32]:
# 1. 데이터가 몇 개나 분석에 사용되었나?
print(f"분석에 사용된 데이터 수: {len(df)}")

# 2. 데이터 기초 통계 (Q, E, S, P, y의 범위 확인)
print("\n[기초 통계량]")
print(df.describe())

# 3. 변수 간 상관관계
print("\n[상관관계 행렬]")
print(df[['Q', 'E', 'S', 'P_adj', 'y']].corr())

분석에 사용된 데이터 수: 92

[기초 통계량]
               Q          E          S          P             y      P_adj
count  92.000000  92.000000  92.000000  92.000000     92.000000  92.000000
mean    0.763695   0.751404   0.801943   0.899873   3824.076087   1.349810
std     0.084146   0.091534   0.083157   0.085828   6530.775600   0.128741
min     0.325000   0.325000   0.325000   0.394481      0.000000   0.591721
25%     0.741854   0.728613   0.788551   0.884006    177.000000   1.326009
50%     0.780503   0.758034   0.815324   0.911180   1052.000000   1.366770
75%     0.810254   0.797407   0.844182   0.947130   4294.750000   1.420695
max     0.926861   0.922002   0.931151   0.997258  34011.000000   1.495886

[상관관계 행렬]
              Q         E         S     P_adj         y
Q      1.000000  0.613853  0.761308  0.024112  0.071165
E      0.613853  1.000000  0.677202  0.356044  0.039492
S      0.761308  0.677202  1.000000  0.247282  0.083950
P_adj  0.024112  0.356044  0.247282  1.000000  0.015008
y     

In [34]:
import requests
import pandas as pd
import statsmodels.api as sm

# 1. 설정
BASE_URL = "https://estranged-simple-unknowing.ngrok-free.dev"
HEADERS = {"ngrok-skip-browser-warning": "true"}

# 분석 대상 5개 카테고리 컬렉션 명시
target_collections = [
    "cream_products",
    "lotion_products",
    "skintoner_products",
    "mist_products",
    "essence_products"
]

all_extracted_data = []

print("📂 Qdrant 데이터 통합 수집 중...")

for col_name in target_collections:
    try:
        scroll_url = f"{BASE_URL}/collections/{col_name}/points/scroll"
        # 각 컬렉션별로 모든 데이터를 가져오기 위해 limit을 넉넉히 설정 (필요시 조절)
        payload = {"limit": 2000, "with_payload": True}
        res = requests.post(scroll_url, json=payload, headers=HEADERS)

        if res.status_code == 200:
            points = res.json()['result']['points']
            print(f"✅ {col_name:<18} | 로드된 데이터: {len(points):>4}개")

            for p in points:
                pay = p['payload']
                # 실제 DB의 Key값(Q_score, sales_index 등)이 대소문자까지 일치해야 함을 주의하세요!
                all_extracted_data.append({
                    'category': col_name,
                    'Q': pay.get('Q_score'),
                    'E': pay.get('E_score'),
                    'S': pay.get('S_score'),
                    'P': pay.get('P_score'),
                    'y': pay.get('sales_index')
                })
        else:
            print(f"❌ {col_name:<18} | 연결 실패 (상태 코드: {res.status_code})")

    except Exception as e:
        print(f"⚠️ {col_name:<18} | 에러 발생: {e}")

# 2. 데이터 정제
df = pd.DataFrame(all_extracted_data)

# 결측치가 있는 행(NaN) 제거 - 분석 신뢰도를 위해 필수
original_len = len(df)
df = df.dropna(subset=['Q', 'E', 'S', 'P', 'y'])
cleaned_len = len(df)

print("-" * 50)
print(f"📊 최종 분석 가용한 데이터 총계: {cleaned_len}개 (누락 제거: {original_len - cleaned_len}개)")
print("-" * 50)

# 3. 회귀분석 실시 (y = b0 + b1*Q + b2*E + b3*S + 1.5*b4*P)
if cleaned_len > 10:
    # 질문자님의 요청대로 P값에 1.5 가중치 선반영
    df['P_adj'] = 1.5 * df['P']

    # 독립변수(X)와 종속변수(y) 설정
    X = df[['Q', 'E', 'S', 'P_adj']]
    X = sm.add_constant(X) # 상수항(Intercept) 추가
    y = df['y']

    # OLS 모델 학습
    model = sm.OLS(y, X).fit()

    # 결과 출력
    print("\n[ 최종 회귀분석 결과 보고서 ]")
    print(model.summary())

    # 가중치 픽스를 위한 요약 멘트 생성
    print("\n" + "="*50)
    print("📢 추천 시스템 적용 가이드")
    coeffs = model.params
    print(f"1. 기본 상수(b0): {coeffs['const']:.4f}")
    print(f"2. 품질(Q) 가중치: {coeffs['Q']:.4f}")
    print(f"3. 가치(E) 가중치: {coeffs['E']:.4f}")
    print(f"4. 감성(S) 가중치: {coeffs['S']:.4f}")
    print(f"5. 가성비(P) 가중치(1.5배 보정값): {coeffs['P_adj']:.4f}")
    print("="*50)

else:
    print("🚨 데이터가 너무 적어 분석을 진행할 수 없습니다. DB의 필드명을 다시 확인해 주세요.")

📂 Qdrant 데이터 통합 수집 중...
✅ cream_products     | 로드된 데이터:   92개
✅ lotion_products    | 로드된 데이터:   24개
✅ skintoner_products | 로드된 데이터:   59개
✅ mist_products      | 로드된 데이터:   18개
✅ essence_products   | 로드된 데이터:  185개
--------------------------------------------------
📊 최종 분석 가용한 데이터 총계: 0개 (누락 제거: 378개)
--------------------------------------------------
🚨 데이터가 너무 적어 분석을 진행할 수 없습니다. DB의 필드명을 다시 확인해 주세요.


In [35]:
# 가장 데이터가 많은 essence_products에서 샘플 하나 확인
test_url = f"{BASE_URL}/collections/essence_products/points/scroll"
test_res = requests.post(test_url, json={"limit": 1, "with_payload": True}, headers=HEADERS)

if test_res.status_code == 200:
    sample_payload = test_res.json()['result']['points'][0]['payload']
    print("📋 DB에 실제 저장된 필드명 목록:")
    print(sample_payload.keys())
    print("\n📋 실제 데이터 내용:")
    print(sample_payload)
else:
    print("샘플 데이터를 가져오지 못했습니다.")

📋 DB에 실제 저장된 필드명 목록:
dict_keys(['olive_id', 'master_id', 'search_text', 'olive_name', 'olive_url', 'category', 'original_price', 'sale_price', 'discount_rate', 'volume', 'olive_review_count', 'olive_rating', 'badges', 'olive_total_ml', 'olive_qty', 'olive_is_promo', 'olive_promo_keyword', 'olive_clean_price', 'olive_price_per_ml', 'naver_name', 'naver_price', 'naver_url', 'naver_review_count', 'naver_total_ml', 'naver_qty', 'naver_is_promo', 'naver_promo_keyword', 'naver_clean_price', 'naver_price_per_ml', 'musinsa_name', 'musinsa_price', 'musinsa_url', 'musinsa_review_count', 'musinsa_total_ml', 'musinsa_qty', 'musinsa_is_promo', 'musinsa_promo_keyword', 'musinsa_clean_price', 'musinsa_price_per_ml', 'naver_id', 'musinsa_id', 'musinsa_image_url', 'naver_image_url', 'olive_image_url', 'Q_mass_total', 'Q_pos_product', 'E_mass_total', 'E_pos_product', 'S_mass_total', 'S_pos_product', 'P_score'])

📋 실제 데이터 내용:
{'olive_id': 'A000000202920', 'master_id': 'PRD_87684185496', 'search_text': '올

In [41]:
import requests
import pandas as pd
import statsmodels.api as sm

BASE_URL = "https://estranged-simple-unknowing.ngrok-free.dev"
HEADERS = {"ngrok-skip-browser-warning": "true"}

target_collections = [
    "cream_products", "lotion_products", "skintoner_products",
    "mist_products", "essence_products"
]

all_extracted_data = []

print("📂 Qdrant 데이터 통합 수집 및 필드 매핑 중...")

for col_name in target_collections:
    try:
        scroll_url = f"{BASE_URL}/collections/{col_name}/points/scroll"
        payload = {"limit": 2000, "with_payload": True}
        res = requests.post(scroll_url, json=payload, headers=HEADERS)

        if res.status_code == 200:
            points = res.json()['result']['points']
            print(f"✅ {col_name:<18} | {len(points):>4}개 로드")

            for p in points:
                pay = p['payload']
                # --- [수정된 필드명 매핑] ---
                all_extracted_data.append({
                    'category': col_name, # 'category' 컬럼 추가
                    'Q': pay.get('Q_pos_product'),
                    'E': pay.get('E_pos_product'),
                    'S': pay.get('S_pos_product'),
                    'P': pay.get('P_score'),
                    'y': pay.get('olive_review_count') # 판매량 대용 지표
                })

    except Exception as e:
        print(f"⚠️ {col_name} 에러: {e}")

# 2. 데이터 정제 및 회귀분석 준비
df = pd.DataFrame(all_extracted_data).dropna()

print("-" * 50)
print(f"📊 분석 가용 데이터: {len(df)}개")
print("-" * 50)

if len(df) > 10:
    # 회귀식: y = b0 + b1Q + b2E + b3S + 1.5b4P
    df['P_adj'] = 1.5 * df['P']

    X = df[['Q', 'E', 'S', 'P_adj']]
    X = sm.add_constant(X)
    y = df['y']

    model = sm.OLS(y, X).fit()

    print("\n[ 최종 회귀분석 결과 ]")
    print(model.summary())

    # 계수 요약
    coeffs = model.params
    print("\n" + "="*50)
    print(f"계수 (b값):")
    print(f" - b0(상수): {coeffs['const']:.4f}")
    print(f" - b1(품질): {coeffs['Q']:.4f}")
    print(f" - b2(가치): {coeffs['E']:.4f}")
    print(f" - b3(감성): {coeffs['S']:.4f}")
    print(f" - b4(가성비): {coeffs['P_adj']:.4f}")
    print("="*50)
else:
    print("데이터가 부족합니다. 필드명 매핑을 다시 확인하세요.")

📂 Qdrant 데이터 통합 수집 및 필드 매핑 중...
✅ cream_products     |   92개 로드
✅ lotion_products    |   24개 로드
✅ skintoner_products |   59개 로드
✅ mist_products      |   18개 로드
✅ essence_products   |  185개 로드
--------------------------------------------------
📊 분석 가용 데이터: 378개
--------------------------------------------------

[ 최종 회귀분석 결과 ]
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                 -0.003
Method:                 Least Squares   F-statistic:                    0.6840
Date:                Tue, 28 Apr 2026   Prob (F-statistic):              0.603
Time:                        17:38:06   Log-Likelihood:                -4046.7
No. Observations:                 378   AIC:                             8103.
Df Residuals:                     373   BIC:                             8123.
Df Model:                           4   

**post-processing : 회귀분석에서 나온 날것의 계수(b)들을 합계가 1(!00%)이 되는 비율로 변환하고, 음수값을 처리함.**
특히 s점수가 마이너스인 점과, 가중치들의 단위가 너무 큰 점을 해결 -> 실제 서비스에 바로 적용 가능한 현실적인 추정 가중치로 변환

In [42]:
import numpy as np

# 1. 회귀 분석에서 나온 원본 계수 (추출된 값 입력)
raw_coeffs = {
    'Q': 7765.8286,
    'E': 885.0202,
    'S': -18086.0943,  # 문제가 된 음수 값
    'P': 8218.7421
}

# 2. 보정 로직 (Business Logic)
# - 음수가 나온 S는 최소 가중치(예: 다른 변수들의 평균의 10%)로 강제 치환하거나 0으로 처리
adjusted_coeffs = {}
for key, val in raw_coeffs.items():
    if key == 'S' and val < 0:
        adjusted_coeffs[key] = 500  # 최소한의 긍정적 영향력 부여 (보정값)
    else:
        adjusted_coeffs[key] = val

# 3. 정규화 (Normalization)
# - 모든 가중치의 합이 1이 되도록 비율로 변환 (추천 공식에 넣기 가장 좋음)
total = sum(adjusted_coeffs.values())
final_weights = {k: v / total for k, v in adjusted_coeffs.items()}

print("--- [최종 추천 시스템용 가중치 비율] ---")
for k, v in final_weights.items():
    print(f"{k} (가중치): {v:.2%}")

# 4. 최종 추천 점수 산출 함수 (예시)
def get_recommendation_score(q, e, s, p):
    score = (final_weights['Q'] * q +
             final_weights['E'] * e +
             final_weights['S'] * s +
             final_weights['P'] * p)
    return score

--- [최종 추천 시스템용 가중치 비율] ---
Q (가중치): 44.71%
E (가중치): 5.10%
S (가중치): 2.88%
P (가중치): 47.32%


회귀결과를 바탕으로 비즈니스 로직 추가 => 최중 가중치 결정 및 스코어링 함수

In [43]:
import pandas as pd

# 1. 회귀분석 결과(coef) 바탕으로 가중치 재설정
# 음수가 나온 S는 최소 비중으로, 비중이 낮았던 E는 보조 비중으로 조정
raw_weights = {
    'Q': 7765.8,
    'E': 885.0,
    'S': 500.0,   # -18086.1 대신 최소 긍정 가중치로 보정
    'P': 8218.7   # 이미 1.5배 가중치가 녹아있는 값
}

# 2. 가중치 정규화 (전체 합이 1이 되도록)
total_weight = sum(raw_weights.values())
norm_weights = {k: v / total_weight for k, v in raw_weights.items()}

def calculate_final_score(row):
    """
    최종 추천 점수(y_hat)를 계산하는 함수
    정규화된 가중치를 사용하여 0~1 사이의 점수를 산출
    """
    # 설계하신 모델: y = b1*Q + b2*E + b3*S + 1.5*b4*P
    # (이미 P_adj를 사용해 b4를 구했으므로 가중치 비율대로 곱함)

    score = (
        row['Q'] * norm_weights['Q'] +    # 'Q_pos_product' 대신 'Q' 사용
        row['E'] * norm_weights['E'] +    # 'E_pos_product' 대신 'E' 사용
        row['S'] * norm_weights['S'] +    # 'S_pos_product' 대신 'S' 사용
        row['P'] * norm_weights['P']     # 'P_score' 대신 'P' 사용
    )
    return score

# 3. 전체 데이터프레임에 적용 (df는 앞서 5개 카테고리를 합친 데이터)
df['final_recommend_score'] = df.apply(calculate_final_score, axis=1)

# 4. 결과 확인: 점수가 높은 상위 5개 상품 추천
top_recommendations = df.sort_values(by='final_recommend_score', ascending=False).head(5)

print("--- [최종 가중치 비율 반영 결과] ---")
for k, v in norm_weights.items():
    print(f"{k} 지수 반영 비중: {v:.2%}")

print("\n--- [추천 시스템 상위 리스트] ---")
print(top_recommendations[['category', 'final_recommend_score']])


--- [최종 가중치 비율 반영 결과] ---
Q 지수 반영 비중: 44.71%
E 지수 반영 비중: 5.10%
S 지수 반영 비중: 2.88%
P 지수 반영 비중: 47.32%

--- [추천 시스템 상위 리스트] ---
               category  final_recommend_score
89       cream_products               0.954220
174  skintoner_products               0.951487
115     lotion_products               0.948754
84       cream_products               0.940897
141  skintoner_products               0.927615
